# Deliverable D: Modeling & Evaluation
**Qiyas Data Science & AI Hackathon - Addis Ride Demand Forecasting Challenge**

This notebook imports **ALL candidate models** in a single file and evaluates them on a **chronological time-split validation set** (Jan 1 - Oct 17 for training, Oct 18 - Oct 31 for validation).

### Included Model Families:
1. **Linear Models**: `LinearRegression`, `Ridge`, `Lasso`
2. **Tree Models**: `DecisionTreeRegressor`, `RandomForestRegressor`
3. **Gradient Boosting Models**: `GradientBoostingRegressor`, `HistGradientBoostingRegressor`
4. **Advanced Gradient Boosting Frameworks**: `XGBRegressor` (XGBoost), `LGBMRegressor` (LightGBM), `CatBoostRegressor` (CatBoost)

In [ ]:
# Single Cell Importing ALL Candidate Models & Metrics
import os, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Linear Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso

# 2. Decision Trees & Ensembles
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)

# 3. Gradient Boosting Frameworks
import xgboost as xgb
from xgboost import XGBRegressor

import lightgbm as lgb
from lightgbm import LGBMRegressor

from catboost import CatBoostRegressor

# 4. Evaluation Metrics & Pipelines
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
    mean_absolute_percentage_error
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Base directory path
base_dir = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if base_dir not in sys.path:
    sys.path.insert(0, base_dir)

from src.features import extract_features, add_historical_lags_and_aggregations

print('SUCCESS: All 10 Machine Learning models and metrics imported in a single file!')

## 1. Load Master Dataset & Prepare Chronological Train/Val Split

In [ ]:
pdir = os.path.join(base_dir, 'data', 'processed')
train_master = pd.read_csv(os.path.join(pdir, 'master_train.csv'))
test_master = pd.read_csv(os.path.join(pdir, 'master_test.csv'))

train_df = extract_features(train_master, is_train=True)
test_df = extract_features(test_master, is_train=False)
train_df, test_df = add_historical_lags_and_aggregations(train_df, test_df)

train_df['pickup_datetime'] = pd.to_datetime(train_df['pickup_datetime'])
split_date = pd.Timestamp('2025-10-18')

train_sub = train_df[(train_df['pickup_datetime'] < split_date) & train_df['trips'].notna()].copy()
val_sub = train_df[(train_df['pickup_datetime'] >= split_date) & train_df['trips'].notna()].copy()

print(f'Train sub shape: {train_sub.shape}')
print(f'Val sub shape:   {val_sub.shape}')

## 2. Train and Benchmark ALL 10 Models in One Execution Loop

In [ ]:
feature_cols = [
    'hour', 'dayofweek', 'day', 'month', 'is_weekend', 'is_payday', 'hour_sin', 'hour_cos',
    'temp_c', 'rain_mm', 'humidity_pct', 'wind_kmh', 'rain_3h_sum', 'rain_class',
    'has_event', 'event_count', 'has_sports', 'has_concert', 'has_public_holiday', 'event_attendance',
    'lag_24h', 'lag_48h', 'lag_168h', 'lag_336h',
    'rolling_24h_mean_trips', 'rolling_24h_std_trips', 'rolling_7d_mean_trips',
    'zone_hour_mean_trips', 'zone_dow_hour_mean_trips'
]
feature_cols = [c for c in feature_cols if c in train_sub.columns]
categorical_cols = ['zone_clean', 'zone_x_rain', 'zone_x_hour']
categorical_cols = [c for c in categorical_cols if c in train_sub.columns]

X_train, y_train = train_sub[feature_cols + categorical_cols], train_sub['trips']
X_val, y_val = val_sub[feature_cols + categorical_cols], val_sub['trips']

models = {
    'CatBoost': CatBoostRegressor(iterations=300, depth=7, learning_rate=0.04, verbose=0, random_seed=42),
    'LightGBM': LGBMRegressor(n_estimators=300, num_leaves=63, learning_rate=0.04, random_state=42, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, max_depth=7, learning_rate=0.04, random_state=42, n_jobs=-1),
    'Hist Gradient Boosting': HistGradientBoostingRegressor(max_iter=300, max_depth=8, learning_rate=0.04, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=6, random_state=42),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'Lasso Regression': Lasso(alpha=0.1),
    'Ridge Regression': Ridge(alpha=1.0),
    'Linear Regression': LinearRegression()
}

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), feature_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

results = []
for name, model in models.items():
    t0 = time.time()
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])
    pipe.fit(X_train, y_train)
    t_diff = round(time.time() - t0, 2)
    
    preds = np.clip(pipe.predict(X_val), 0, None)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    mae = mean_absolute_error(y_val, preds)
    r2 = r2_score(y_val, preds)
    
    results.append({'Model Name': name, 'Validation RMSE': round(rmse, 4), 'Validation MAE': round(mae, 4), 'R2 Score': round(r2, 4), 'Train Time (s)': t_diff})

leaderboard = pd.DataFrame(results).sort_values('Validation RMSE')
display(leaderboard)